# 柔性资源约束项目调度问题

**类别：** 调度

来源：[https://www.hexaly.com/templates/flexible-resource-constrained-project-scheduling-problem](https://www.hexaly.com/templates/flexible-resource-constrained-project-scheduling-problem)


## 问题

**在柔性资源约束项目调度问题**中，一个项目由一组需要调度的任务组成。每个任务有一组兼容的资源，并且必须由其中一个资源处理。每个任务的处理时间和资源使用量（也称为权重）取决于其选择的资源。每个资源都有一个给定的最大容量：它可以同时处理多个任务，但所处理任务的权重之和不能超过该最大容量。任务之间还存在紧前约束：每个任务必须在其任意紧后任务开始之前结束。目标是找到一个使 makespan（即所有任务处理完成的时间）最小化的调度方案。

	

### 学到的建模原则

- 添加 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模任务到资源的分配
- 添加 [interval decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模任务
- 定义嵌套的 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来建模柔性累计资源约束


## 数据

我们提供的**柔性资源约束项目调度问题**实例遵循 Patterson [[1]](#footnote-1) 格式：

- 第一行：

- 任务数
- 可更新资源数
- 第二行：每个资源的最大容量
- 从第三行开始，对每个任务和每个资源：

- 任务在该资源上的处理时间
- 资源使用量（权重）
- 从下一行开始，对每个任务：

- 紧后任务的数量
- 每个紧后任务的 ID


## 模型

柔性资源约束项目调度问题的Hexaly模型使用表示任务的 interval decision variables，以及表示每个资源上调度任务集合的 set decision variables。

使用 [**partition**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition) 算子，我们确保每个任务被分配到恰好一个资源。对于每个任务，我们借助 **contains** 算子过滤掉不兼容的资源。然后使用 [**find**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#find) 算子检索为处理每个任务所选择的资源的索引。这使我们能够推导每个任务的处理时间和权重（它们取决于所选的资源），并相应地约束每个 interval 的长度。

然后我们编写紧前约束：每个任务必须在其任意紧后任务开始之前结束。

累计资源约束可以表述如下：对于每个资源以及每个时间槽 t，正在被处理的任务所消耗的资源量不得超过该资源的容量。为了对这些约束建模，我们对每个资源和每个时间槽，将所有活动任务的权重求和。由于每个资源上调度任务的集合不是固定的，并且在搜索过程中会发生变化，因此这些求和通过 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 计算。我们将变参 **and** 与另一个 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 结合使用，以确保在任何时刻都满足资源容量约束。借助此变参 **and**，即使时间跨度非常大，约束的表述仍然紧凑且高效。

需要最小化的 makespan 是所有任务结束的时间。

[1] Patterson, J. H.,( 1984), [A comparison of exact approaches for solving the multiple constrained resource, Project Scheduling Problem](https://doi.org/10.1287/mnsc.30.7.854), Management Science, Vol. 30, p854-86


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


def read_instance(filename):
    with open(filename) as f:
        lines = f.readlines()

    first_line = lines[0].split()

    # Number of tasks
    nb_tasks = int(first_line[0])

    # Number of resources
    nb_resources = int(first_line[1])

    # Maximum capacity of each resource
    capacity = [int(lines[1].split()[r]) for r in range(nb_resources)]

    # Duration of task i if task i is done by resource r
    task_processing_time_data = [[] for i in range(nb_tasks)]

    # Resource weight of resource r required for task i
    weight = [[] for r in range(nb_resources)]

    # Number of successors
    nb_successors = [0 for i in range(nb_tasks)]

    # Successors of each task i
    successors = [[] for i in range(nb_tasks)]

    for i in range(nb_tasks):
        line_d_w = lines[i + 2].split()
        for r in range(nb_resources):
            task_processing_time_data[i].append(int(line_d_w[2 * r]))
            weight[r].append(int(line_d_w[2 * r + 1]))

        line_succ = lines[i + 2 + nb_tasks].split()
        nb_successors[i] = int(line_succ[0])
        successors[i] = [int(elm) for elm in line_succ[1::]]

    # Trivial upper bound for the end times of the tasks
    horizon = sum(max(task_processing_time_data[i][r] for r in range(nb_resources)) for i in range(nb_tasks))

    return (nb_tasks, nb_resources, capacity, task_processing_time_data, weight, nb_successors, successors, horizon)


def main(instance_file, output_file, time_limit):
    nb_tasks, nb_resources, capacity, task_processing_time_data, weight,\
        nb_successors, successors, horizon = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Set of tasks done by each resource
        resources_tasks = [model.set(nb_tasks) for r in range(nb_resources)]
        resources = model.array(resources_tasks)

        # Only compatible resources can be selected for a task
        for i in range(nb_tasks):
            for r in range(nb_resources):
                if task_processing_time_data[i][r] == 0 and weight[r][i] == 0:
                    model.constraint(model.contains(resources_tasks[r], i) == 0)

        # For each task, the selected resource
        task_resource = [model.find(resources, t) for t in range(nb_tasks)]

        # All tasks are scheduled on the resources
        model.constraint(model.partition(resources))

        # Interval decisions: time range of each task
        tasks = [model.interval(0, horizon) for i in range(nb_tasks)]

        # Create Hexaly arrays to be able to access them with an "at" operator
        tasks_array = model.array(tasks)
        task_processing_time = model.array(task_processing_time_data)
        weight_array = model.array(weight)

        # Task duration constraints
        for i in range(nb_tasks):
            model.constraint(model.length(tasks[i]) == task_processing_time[i][task_resource[i]])

        # Precedence constraints between the tasks
        for i in range(nb_tasks):
            for s in range(nb_successors[i]):
                model.constraint(tasks[i] < tasks[successors[i][s]])

        # Makespan: end of the last task
        makespan = model.max([model.end(tasks[i]) for i in range(nb_tasks)])

        # Cumulative resource constraints
        for r in range(nb_resources):
            capacity_respected = model.lambda_function(
                lambda t: model.sum(resources_tasks[r], model.lambda_function(
                    lambda i: model.at(weight_array, r, i) * model.contains(tasks_array[i], t)))
                <= capacity[r])
            model.constraint(model.and_(model.range(makespan), capacity_respected))

        # Minimize the makespan
        model.minimize(makespan)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        # - total makespan
        # - for each task, the task id, the selected resource, the start and end times
        #
        if output_file != None:
            with open(output_file, "w") as f:
                print("Solution written in file", output_file)
                f.write(str(makespan.value) + "\n")
                for i in range(nb_tasks):
                    f.write(
                        str(i) + " " + str(task_resource[i].value) + " " + str(tasks[i].value.start()) + " " +
                        str(tasks[i].value.end()))
                    f.write("\n")


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python flexible_cumulative.py instance_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    main(instance_file, output_file, time_limit)
